In [1]:
# Code to consolidate every folder, text and image and combine them to make a dataset

import json
import glob
import os
import shutil

# --- CONFIGURATION ---
OUTPUT_JSON = "final_war_dataset_master.json"
MASTER_IMAGE_FOLDER = "master_images"
# This is the directory where all your files from the screenshot are located
# If the script is IN that folder, use "."
BASE_DIR = "." 

if not os.path.exists(MASTER_IMAGE_FOLDER):
    os.makedirs(MASTER_IMAGE_FOLDER)

# Use your expanded keyword list
STRICT_WAR_KEYWORDS = [
    "war", "conflict", "missile", "attack", "strike", "military", "irgc", "centcom",
    "strait of hormuz", "nuclear", "drone", "hezbollah", "houthi", "israel", "iran",
    "russia", "ukraine", "gaza", "airstrike", "invasion", "trump", "soleimani",
    "india", "pakistan", "loc", "line of control", "kashmir", "balakot", "pulwama",
    "surgical strike", "border clash", "idf", "iron dome", "hamas", "west bank",
    "jerusalem", "quds force", "ayatollah", "shia militia", "proxy war", "donbas",
    "crimea", "kyiv", "mariupol", "kharkiv", "nato", "sanctions", "mobilization",
    "frontline", "occupation", "ceasefire", "escalation", "casualties", "shelling",
    "artillery", "rebels", "terrorist attack", "peace talks", "blockade", "special forces",
    "tanks", "fighter jets", "ballistic missile", "hypersonic", "naval blockade",
    "air defense"
]

def find_matching_folder(json_filename):
    """
    Tries to find a folder that shares a suffix with the JSON file.
    Example: war_dataset_large_ayu1 -> downloaded_images_ayu1
    """
    # Extract the suffix (e.g., 'ayu1', 'pramit', 'ayu2')
    parts = json_filename.replace(".json", "").split("_")
    suffix = parts[-1] 
    
    all_folders = [d for d in os.listdir(BASE_DIR) if os.path.isdir(os.path.join(BASE_DIR, d))]
    for folder in all_folders:
        if suffix in folder and folder != MASTER_IMAGE_FOLDER:
            return folder
    return None

def is_genuine_content(article):
    content = article.get("content") or article.get("text") or ""
    title = article.get("title") or ""
    content_lower = content.lower()
    
    red_flags = ["subscribe to read", "sign in to continue", "enable javascript", "watch the video"]
    if any(flag in content_lower for flag in red_flags): return False
    
    if 100 <= len(content) < 350:
        urgent = ["breaking", "missile", "explosion", "strike", "urgent"]
        return any(word in title.lower() for word in urgent)
        
    return len(content) >= 100

def clean_and_merge():
    combined_data = []
    seen_urls = set()
    master_id = 1
    
    json_files = glob.glob(os.path.join(BASE_DIR, "*.json"))

    print(f"🚀 Processing {len(json_files)} datasets...")

    for file_path in json_files:
        json_name = os.path.basename(file_path)
        if json_name == OUTPUT_JSON: continue
        
        # 1. Identify the specific image folder for this user
        matching_folder = find_matching_folder(json_name)
        if matching_folder:
            print(f"📦 Mapping {json_name} -> /{matching_folder}")
        else:
            print(f"📝 Mapping {json_name} -> (No Image Folder Found)")

        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                for article in data:
                    url = article.get("url")
                    if not url or url in seen_urls: continue

                    if is_genuine_content(article):
                        content = article.get("content") or article.get("text") or ""
                        title = article.get("title") or ""
                        
                        if any(word in content.lower() or word in title.lower() for word in STRICT_WAR_KEYWORDS):
                            
                            # --- MULTIMODAL LINKING ---
                            old_img_name = article.get("local_image") or article.get("local_image_path") or article.get("top_image")
                            new_img_name = "N/A"

                            if matching_folder and old_img_name and old_img_name != "N/A":
                                # Clean the path (sometimes scrapers save full paths, we just want the filename)
                                clean_img_filename = os.path.basename(old_img_name)
                                full_old_path = os.path.join(BASE_DIR, matching_folder, clean_img_filename)

                                if os.path.exists(full_old_path):
                                    ext = os.path.splitext(clean_img_filename)[1]
                                    new_img_name = f"war_data_{master_id}{ext}"
                                    shutil.copy(full_old_path, os.path.join(MASTER_IMAGE_FOLDER, new_img_name))

                            combined_data.append({
                                "id": master_id,
                                "source": article.get("source", "unknown"),
                                "title": title,
                                "url": url,
                                "date": article.get("date") or "N/A",
                                "content": content,
                                "local_image": new_img_name
                            })
                            seen_urls.add(url)
                            master_id += 1
        except Exception as e:
            print(f"Error in {json_name}: {e}")

    # Save Master File
    with open(OUTPUT_JSON, "w", encoding='utf-8') as f:
        json.dump(combined_data, f, indent=4, ensure_ascii=False)

    print("-" * 30)
    print(f"✅ Master Dataset Created: {len(combined_data)} unique articles.")
    print(f"All valid images consolidated in /{MASTER_IMAGE_FOLDER}")

if __name__ == "__main__":
    clean_and_merge()

🚀 Processing 5 datasets...
📦 Mapping war_dataset_large_ayu1.json -> /downloaded_images_ayu1
📦 Mapping war_dataset_v3_new_pramit.json -> /warimg_new_pramit
📦 Mapping war_dataset_v4_ayu2.json -> /war_images_ayu2
📦 Mapping war_dataset_v5_ayu3.json -> /war_images_two_ayu3
📝 Mapping war_data_formatted_adri.json -> (No Image Folder Found)
------------------------------
✅ Master Dataset Created: 901 unique articles.
All valid images consolidated in /master_images


In [2]:
# Some unrelated articles may have entered the list, further processing being done here

import json
import os
import shutil

# --- NEW: THE EXCLUSION LIST ---
# If these words appear, the article is likely NOT about the actual war front
EXCLUSION_KEYWORDS = [
    "bollywood", "hollywood", "actor", "actress", "movie", "film", "cinema",
    "sensex", "nifty", "stock market", "trading", "investment portfolio",
    "cricket", "ipl", "football", "match score", "zodiac", "horoscope",
    "lifestyle", "fashion", "skincare", "wedding", "divorce"
]

STRICT_WAR_KEYWORDS = [
    "war", "conflict", "missile", "attack", "strike", "military", "irgc", "centcom",
    "strait of hormuz", "nuclear", "drone", "hezbollah", "houthi", "israel", "iran",
    "russia", "ukraine", "gaza", "airstrike", "invasion", "trump", "soleimani",
    "idf", "iron dome", "hamas", "west bank", "frontline", "artillery", "jets"
]

def is_actually_war(title, content):
    text = (title + " " + content).lower()
    
    # 1. IMMEDIATE REJECTION
    # If it contains an exclusion word, it's out.
    if any(word in text for word in EXCLUSION_KEYWORDS):
        return False

    # 2. SELECTIVE FILTERING
    # We want to avoid "Oil found" or "Stock crashes"
    # An article is only "War" if it has at least TWO war keywords 
    # OR one very specific word like 'missile' or 'irgc'
    matches = [word for word in STRICT_WAR_KEYWORDS if word in text]
    
    # High-intensity words that prove it's a war story
    high_intensity = ["missile", "airstrike", "invasion", "irgc", "houthi", "iron dome", "frontline"]
    
    if any(hi in text for hi in high_intensity):
        return True
    
    return len(matches) >= 2 # Otherwise, it needs at least 2 general keywords

def final_refine():
    # Load your current "dirty" master file
    input_file = "final_war_dataset_master.json"
    output_file = "final_war_dataset_CLEANED.json"
    
    if not os.path.exists(input_file):
        print("Error: Master file not found.")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    refined_data = []
    removed_count = 0

    print(f"Starting refinement on {len(data)} articles...")

    for article in data:
        title = article.get("title", "")
        content = article.get("content", "")
        
        if is_actually_war(title, content):
            refined_data.append(article)
        else:
            removed_count += 1
            # Optional: Remove the image file from master_images if the article is rejected
            img_name = article.get("local_image")
            if img_name and img_name != "N/A":
                img_path = os.path.join("master_images", img_name)
                if os.path.exists(img_path):
                    os.remove(img_path)

    # Re-index IDs so they are sequential again
    for i, article in enumerate(refined_data):
        article["id"] = i + 1

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(refined_data, f, indent=4, ensure_ascii=False)

    print(f"✅ Refinement Complete!")
    print(f"Articles kept: {len(refined_data)}")
    print(f"Articles removed: {removed_count}")

if __name__ == "__main__":
    final_refine()

Starting refinement on 901 articles...
✅ Refinement Complete!
Articles kept: 417
Articles removed: 484


In [3]:
# Again content duplication may happen but different URLs, so further cleanup being done here

import json
import os
import hashlib

# 1. THE REJECTION LIST (Add anything you see in your 'bad' articles here)
HARD_EXCLUSIONS = [
    "sensex", "nifty", "equity", "stock market", "dividend", "investment",
    "bollywood", "hollywood", "actor", "actress", "film", "cinema", "box office",
    "cricket", "ipl", "football", "horoscope", "lifestyle", "skincare", "recipe",
    "discount", "deals", "amazon", "flipkart", "tech news", "smartphone"
]

# 2. THE WAR LIST (Only keep if these are in the TITLE or high density in content)
WAR_LIST = [
    "war", "conflict", "missile", "strike", "attack", "military", "irgc", "centcom",
    "houthi", "hezbollah", "israel", "iran", "russia", "ukraine", "gaza", "invasion"
]

def clean_and_deduplicate():
    input_file = "final_war_dataset_master.json"
    output_file = "final_war_dataset_PERFECT.json"
    
    if not os.path.exists(input_file):
        print("Master file not found.")
        return

    with open(input_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    final_data = []
    seen_content_hashes = set() # For content-based deduplication
    removed_count = 0

    print(f"Starting final scrub of {len(data)} articles...")

    for article in data:
        title = article.get("title", "").lower()
        content = article.get("content", "").lower()
        
        # --- STEP 1: DEDUPLICATION BY CONTENT ---
        # We hash the first 200 chars to see if the story is identical
        content_snippet = content[:200].strip()
        content_hash = hashlib.md5(content_snippet.encode()).hexdigest()
        
        if content_hash in seen_content_hashes:
            removed_count += 1
            continue

        # --- STEP 2: HARD EXCLUSION CHECK ---
        if any(bad in title or bad in content for bad in HARD_EXCLUSIONS):
            removed_count += 1
            continue

        # --- STEP 3: TITLE VALIDATION ---
        # Most "Unrelated" stories have war keywords in the body but NOT the title.
        # This requires the title to actually be about the war.
        if not any(war in title for war in WAR_LIST):
            # Exception: Keep if very high density in content (3+ mentions)
            matches = [w for w in WAR_LIST if w in content]
            if len(matches) < 3:
                removed_count += 1
                continue

        # If it passed everything, keep it
        final_data.append(article)
        seen_content_hashes.add(content_hash)

    # Re-index
    for i, article in enumerate(final_data):
        article["id"] = i + 1

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(final_data, f, indent=4, ensure_ascii=False)

    print("-" * 30)
    print(f"✅ Scrubbing Complete!")
    print(f"Final Count: {len(final_data)}")
    print(f"Trash Removed: {removed_count}")

if __name__ == "__main__":
    clean_and_deduplicate()

Starting final scrub of 901 articles...
------------------------------
✅ Scrubbing Complete!
Final Count: 333
Trash Removed: 568


In [17]:
import json
import os
import shutil
import hashlib

# --- CONFIG ---
MASTER_JSON = "final_war_dataset_PERFECT.json"
MASTER_IMG_DIR = "master_images"
X_DATA_JSON = "master_dataset_cleaned_megha.json" 
X_IMG_DIR = "twitter_images_megha"               

if not os.path.exists(MASTER_IMG_DIR):
    os.makedirs(MASTER_IMG_DIR)

def get_content_hash(text):
    if not text: return "empty"
    return hashlib.md5(text.strip().lower().encode()).hexdigest()

def integrate_x_hex_fix():
    if not os.path.exists(MASTER_JSON):
        print("❌ Master JSON not found.")
        return

    with open(MASTER_JSON, 'r', encoding='utf-8') as f:
        master_data = json.load(f)

    # 1. RESET: Wipe old X data to allow re-importing with images
    print(f"🧹 Cleaning previous X entries...")
    master_data = [item for item in master_data if item.get("source", "").lower() != "x"]

    if not os.path.exists(X_DATA_JSON):
        print(f"❌ X JSON '{X_DATA_JSON}' not found.")
        return

    with open(X_DATA_JSON, 'r', encoding='utf-8') as f:
        x_raw_data = json.load(f)

    if not os.path.exists(X_IMG_DIR):
        print(f"❌ Image folder '{X_IMG_DIR}' not found!")
        return

    # Create a lookup set of all files in the folder for speed
    available_images = os.listdir(X_IMG_DIR)
    
    seen_urls = {item.get("url") for item in master_data if item.get("url") and item.get("url") != "N/A"}
    seen_hashes = {get_content_hash(item.get("content")) for item in master_data}
    
    current_id = max([item.get("id", 0) for item in master_data]) + 1 if master_data else 1
    added_count = 0
    img_success = 0

    print(f"🔄 Processing {len(x_raw_data)} hex-id tweets...")

    for tweet in x_raw_data:
        # Use the correct keys based on your sample: 'text' and 'id'
        content = tweet.get("text") or tweet.get("clean_text") or ""
        tweet_id = str(tweet.get("id", ""))
        content_hash = get_content_hash(content)

        # Skip duplicates
        if content_hash in seen_hashes:
            continue

        # --- IMAGE MATCHING BASED ON HEX ID ---
        new_img_name = "N/A"
        
        if tweet_id:
            # We look for a file that matches the ID (e.g., b2dc4710.jpg)
            match = None
            for img_file in available_images:
                # Check if the ID is in the filename
                if tweet_id in img_file:
                    match = img_file
                    break
            
            if match:
                src_path = os.path.join(X_IMG_DIR, match)
                ext = os.path.splitext(match)[1] or ".jpg"
                new_img_name = f"war_data_x_{current_id}{ext}"
                shutil.copy(src_path, os.path.join(MASTER_IMG_DIR, new_img_name))
                img_success += 1

        # Save to Master
        master_data.append({
            "id": current_id,
            "source": "X",
            "title": f"Tweet {tweet_id}",
            "url": "N/A", # These samples don't show a direct URL key
            "date": "N/A",
            "content": content,
            "local_image": new_img_name
        })
        
        seen_hashes.add(content_hash)
        current_id += 1
        added_count += 1

    with open(MASTER_JSON, "w", encoding='utf-8') as f:
        json.dump(master_data, f, indent=4, ensure_ascii=False)

    print("-" * 30)
    print(f"✅ INTEGRATION SUCCESSFUL")
    print(f"Added Tweets: {added_count}")
    print(f"Images Successfully Caught: {img_success}")
    print(f"Master Images Folder: {MASTER_IMG_DIR}")

if __name__ == "__main__":
    integrate_x_hex_fix()

🧹 Cleaning previous X entries...
🔄 Processing 930 hex-id tweets...
------------------------------
✅ INTEGRATION SUCCESSFUL
Added Tweets: 928
Images Successfully Caught: 928
Master Images Folder: master_images


In [16]:
# Remove X data (IGNORE THIS)

import json

MASTER_JSON = "final_war_dataset_PERFECT.json"

def remove_x_data():
    try:
        with open(MASTER_JSON, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # Keep only articles that are NOT from source "X"
        cleaned_data = [item for item in data if item.get("source", "").lower() != "x"]
        
        with open(MASTER_JSON, 'w', encoding='utf-8') as f:
            json.dump(cleaned_data, f, indent=4, ensure_ascii=False)
            
        print(f"🗑️ Reset Complete. Removed X entries. New Master Size: {len(cleaned_data)}")
    except Exception as e:
        print(f"Error: {e}")

remove_x_data()

🗑️ Reset Complete. Removed X entries. New Master Size: 333


In [1]:
# Final dataset audit

import json
import os

# --- CONFIG ---
MASTER_JSON = "final_war_dataset_PERFECT.json"
MASTER_IMG_DIR = "master_images"

def audit_dataset():
    if not os.path.exists(MASTER_JSON):
        print(f"❌ Error: {MASTER_JSON} not found.")
        return

    with open(MASTER_JSON, 'r', encoding='utf-8') as f:
        data = json.load(f)

    total_articles = len(data)
    source_counts = {}
    articles_with_images = 0
    missing_images = []

    # Get a list of files actually in the folder to verify against JSON
    if os.path.exists(MASTER_IMG_DIR):
        actual_files = set(os.listdir(MASTER_IMG_DIR))
    else:
        actual_files = set()
        print(f"⚠️ Warning: {MASTER_IMG_DIR} folder does not exist!")

    for article in data:
        # Count by Source
        src = article.get("source", "Unknown")
        source_counts[src] = source_counts.get(src, 0) + 1

        # Check Image Integrity
        img_name = article.get("local_image") or article.get("top_image")
        if img_name and img_name != "N/A":
            if img_name in actual_files:
                articles_with_images += 1
            else:
                missing_images.append(article.get("id"))

    # --- DISPLAY RESULTS ---
    print("="*40)
    print("MASTER DATASET AUDIT REPORT")
    print("="*40)
    print(f"Total Unique Articles: {total_articles}")
    print(f"Articles with Images: {articles_with_images}")
    print(f"Articles missing Images: {total_articles - articles_with_images}")
    
    print("\nBREAKDOWN BY SOURCE:")
    for src, count in sorted(source_counts.items(), key=lambda x: x[1], reverse=True):
        print(f" - {src}: {count} articles")

    if missing_images:
        print(f"\nAlert: {len(missing_images)} articles reference images that don't exist in the folder.")
        if len(missing_images) <= 10:
            print(f"   Missing IDs: {missing_images}")
    
    print("="*40)
    print(f"Total files in /{MASTER_IMG_DIR}: {len(actual_files)}")
    print("="*40)

if __name__ == "__main__":
    audit_dataset()

MASTER DATASET AUDIT REPORT
Total Unique Articles: 1261
Articles with Images: 1181
Articles missing Images: 80

BREAKDOWN BY SOURCE:
 - X: 928 articles
 - moneycontrol: 38 articles
 - times_of_india: 36 articles
 - aljazeera: 26 articles
 - military_com: 26 articles
 - bbc: 26 articles
 - theguardian: 22 articles
 - cnn_world: 20 articles
 - army_tech: 17 articles
 - the_hindu: 14 articles
 - defense_one: 13 articles
 - defense_news: 12 articles
 - long_war_journal: 12 articles
 - scmp_intl: 12 articles
 - breaking_defense: 11 articles
 - middle_east_eye: 11 articles
 - military_times: 10 articles
 - rferl: 9 articles
 - defense_news_intl: 7 articles
 - naval_news: 3 articles
 - moneycontrol_news: 3 articles
 - aljazeera_war: 2 articles
 - fox: 1 articles
 - bbc_world: 1 articles
 - the_print: 1 articles

Alert: 16 articles reference images that don't exist in the folder.
Total files in /master_images: 1274
